# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
from pathlib import Path
import numpy as np, pandas as pd, torch, timm

# Tự động tìm gốc repo và nạp thư mục code vào sys.path
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'eval.py').exists():
    REPO_ROOT = REPO_ROOT.parent

CODE_DIR = REPO_ROOT / 'submissions' / '2A202602873_thai_phuc_tien' / 'code'
for p in [REPO_ROOT, CODE_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print('python', platform.python_version(), '| torch', torch.__version__, '| timm', timm.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU')


### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
# import hashlib

# os.makedirs("data/labels", exist_ok=True)
# if not os.path.exists("data/images.zip"):
#     !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

# h = hashlib.md5()
# with open("data/images.zip", "rb") as f:
#     for chunk in iter(lambda: f.read(1 << 20), b""):
#         h.update(chunk)
# assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

# !unzip -q -n data/images.zip -d data/
# !ls data | head

# BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
# for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
#     !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
# !ls -la data/labels

In [ ]:
IMAGES_DIR = "data/images"   # TODO: kiểm tra đúng thư mục chứa các file .jpg sau khi giải nén
LABELS_DIR = "data/labels"

In [ ]:
import os
from pathlib import Path

# Tạo thư mục data trong working directory
os.makedirs("data", exist_ok=True)

# Tạo symbolic link trỏ đến dataset trên Kaggle
!ln -sfn /kaggle/input/datasets/imsparsh/deepweeds/images data/images
!ln -sfn /kaggle/input/datasets/imsparsh/deepweeds/labels data/labels

# Gán biến cho các cell notebook
IMAGES_DIR = "data/images"
LABELS_DIR = "data/labels"
images_path = Path("data/images")
labels_path = Path("data/labels")

# Kiểm tra nhanh
!ls data/labels


## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from dataset import CLASS_NAMES, check_split, load_split

# Định vị thư mục (hỗ trợ cả chạy từ gốc repo hoặc từ thư mục code/)
labels_path = Path(LABELS_DIR if "LABELS_DIR" in globals() else "data/labels")
images_path = Path(IMAGES_DIR if "IMAGES_DIR" in globals() else "data/images")
if not labels_path.exists():
    labels_path = Path("../../data/labels")
    images_path = Path("../../data/images")

out_dir = Path("curves")
out_dir.mkdir(parents=True, exist_ok=True)

# 1. Đọc và kiểm tra split (S1-S4)
train_df, val_df, test_df = load_split(labels_path, fold=0)
stats = check_split(train_df, val_df, test_df, images_path)

print("=== SỐ LƯỢNG ẢNH MỖI TẬP ===")
for split_name, count in stats["n"].items():
    print(f"{split_name.capitalize():<6}: {count} ảnh")

print("\n=== ĐỐI CHIẾU SỐ LƯỢNG MỖI LỚP (Table 1 bài báo) ===")
df_per_class = pd.DataFrame(stats["per_class"])
df_per_class.index = [CLASS_NAMES[i] for i in df_per_class.index]
df_per_class["Total"] = df_per_class.sum(axis=1)
df_per_class["Paper_Table1"] = [1125, 1064, 1031, 1022, 1062, 1009, 1074, 1016, 9106]
df_per_class["Diff"] = df_per_class["Total"] - df_per_class["Paper_Table1"]
print(df_per_class)

# 2. Vẽ biểu đồ phân bố 9 lớp
labels_full = pd.concat([train_df, val_df, test_df])
counts = labels_full["Label"].value_counts().sort_index()

plt.figure(figsize=(10, 4))
plt.bar([CLASS_NAMES[i] for i in counts.index], counts.values, color="steelblue")
plt.xticks(rotation=30, ha="right")
plt.ylabel("Số lượng ảnh")
plt.title(f"Phân bố 9 lớp DeepWeeds (Tổng: {len(labels_full)} ảnh, Negative: {counts[8]/len(labels_full)*100:.1f}%)")
plt.tight_layout()
plt.savefig(out_dir / "class_distribution.png", dpi=200)
plt.show()

# 3. Trực quan hóa 3 ảnh mỗi lớp và kiểm tra kích thước
fig, axes = plt.subplots(len(CLASS_NAMES), 3, figsize=(6, 18))
for c_idx, c_name in enumerate(CLASS_NAMES):
    samples = labels_full[labels_full["Label"] == c_idx].head(3)
    for i, (_, row) in enumerate(samples.iterrows()):
        img_path = images_path / row["Filename"]
        img = Image.open(img_path)
        assert img.size == (256, 256), f"Kích thước sai: {img.size}"
        axes[c_idx, i].imshow(img)
        axes[c_idx, i].axis("off")
        if i == 0:
            axes[c_idx, i].set_title(c_name, loc="left", fontsize=9)

plt.tight_layout()
plt.savefig(out_dir / "sample_classes.png", dpi=200)
plt.show()


In [ ]:
import sys
from pathlib import Path
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'eval.py').exists():
    REPO_ROOT = REPO_ROOT.parent
CODE_DIR = REPO_ROOT / 'submissions' / '2A202602873_thai_phuc_tien' / 'code'
for p in [REPO_ROOT, CODE_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# TODO: kiểm tra pipeline (slide trang 59, GUIDE.md mục 1.3):
#   1) cố định seed
#   2) loss ban đầu của head mới xấp xỉ -ln(1/9) = 2.197
#   3) overfit được một batch nhỏ tới loss gần 0
#   4) vẽ vài ảnh sau augmentation (đã giải chuẩn hoá) cùng nhãn để chắc ảnh và nhãn khớp nhau

import math
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from dataset import IMAGENET_MEAN, IMAGENET_STD, CLASS_NAMES, build_transforms, load_split, make_loader
import model as md
from train import set_seed

# Đảm bảo có sẵn đường dẫn và dữ liệu nếu chạy riêng cell này
img_p = Path(images_path if "images_path" in globals() else "data/images")
if not img_p.exists():
    img_p = Path("../../data/images")

if "train_df" not in globals():
    lbl_p = Path(labels_path if "labels_path" in globals() else "data/labels")
    if not lbl_p.exists():
        lbl_p = Path("../../data/labels")
    train_df, _, _ = load_split(lbl_p, fold=0)

# 1. Cố định Seed
set_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# 2. Kiểm tra Initial Loss (xấp xỉ -ln(1/9) ≈ 2.197)
net = md.build_model("resnet50", pretrained=True, num_classes=9, init="finetune").to(device)
criterion = nn.CrossEntropyLoss()

dummy_x = torch.randn(8, 3, 224, 224, device=device)
dummy_y = torch.randint(0, 9, (8,), device=device)

with torch.no_grad():
    init_logits = net(dummy_x)
    init_loss = criterion(init_logits, dummy_y).item()

expected_loss = -math.log(1.0 / 9.0)
print(f"\n[Test 1] Initial Loss: {init_loss:.4f} (Kỳ vọng: {expected_loss:.4f})")
assert abs(init_loss - expected_loss) < 0.5, f"Lỗi: Initial loss lệch quá xa ({init_loss})"
print("=> Initial Loss: PASS")

# 3. Kiểm tra Overfit 1 batch nhỏ (4-8 mẫu) về loss sát 0 (< 0.05)
optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3)
print("\n[Test 2] Overfitting 1 small batch (8 samples)...")
for step in range(1, 81):
    net.train()
    optimizer.zero_grad()
    outputs = net(dummy_x)
    loss = criterion(outputs, dummy_y)
    loss.backward()
    optimizer.step()
    if step % 20 == 0:
        print(f"Step {step:2d} | Loss: {loss.item():.4f}")

assert loss.item() < 0.05, f"Lỗi: Không overfit được batch nhỏ, loss còn {loss.item():.4f}"
print("=> Overfit Small Batch: PASS")

# 4. Trực quan hóa ảnh sau Augmentation (đã giải chuẩn hóa)
train_tf = build_transforms(train=True, img_size=224, aug="basic")
train_loader = make_loader(train_df, img_p, train_tf, batch_size=4, train=True)
images, labels, filenames = next(iter(train_loader))

mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    unnorm_img = images[i].cpu() * std + mean
    unnorm_img = torch.clamp(unnorm_img, 0.0, 1.0).permute(1, 2, 0).numpy()
    axes[i].imshow(unnorm_img)
    axes[i].set_title(f"{CLASS_NAMES[labels[i]]}", fontsize=10)
    axes[i].axis("off")
plt.suptitle("Kiểm tra ảnh và nhãn sau Augmentation", y=1.05)
plt.tight_layout()
plt.show()
print("=> Augmentation check: PASS")


## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
import sys, gc, time
from pathlib import Path
import pandas as pd, torch, timm

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'eval.py').exists():
    REPO_ROOT = REPO_ROOT.parent
CODE_DIR = REPO_ROOT / 'submissions' / '2A202602873_thai_phuc_tien' / 'code'
for p in [REPO_ROOT, CODE_DIR]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from train import Config, run
from benchmark import latency_report
import model as md

BACKBONES = [
    {"exp_id": "B01", "name": "resnet50", "family": "ResNet (Baseline)"},
    {"exp_id": "B02", "name": "convnext_tiny", "family": "ConvNeXt (Modern CNN)"},
    {"exp_id": "B03", "name": "swin_tiny_patch4_window7_224", "family": "Transformer (Swin)"},
    {"exp_id": "B04", "name": "efficientnet_b0", "family": "Lightweight (EfficientNet)"},
    {"exp_id": "B05", "name": "mobilenetv3_large_100", "family": "Lightweight (MobileNetV3)"},
]

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}\n")
results_backbones = []

for b in BACKBONES:
    exp_id = b["exp_id"]
    name = b["name"]
    print("=" * 60)
    print(f"Đang chạy: {exp_id} - {name} ({b['family']})")
    print("=" * 60)

    cfg = Config(
        exp_id=exp_id,
        backbone=name,
        epochs=12,
        batch_size=64,   # Cấu hình chuẩn cho Kaggle
        num_workers=2,
        seed=0,
        save_test_predictions=False,
    )

    t_start = time.time()
    res = run(cfg)
    train_time = time.time() - t_start

    hist_file = Path(cfg.out_dir) / exp_id / f"seed{cfg.seed}" / "history.csv"
    best_val_top1 = 0.0
    if hist_file.exists():
        df_hist = pd.read_csv(hist_file)
        best_row = df_hist[df_hist["epoch"] == res["best_epoch"]]
        if not best_row.empty and "val_top1" in best_row.columns:
            best_val_top1 = float(best_row["val_top1"].values[0])

    try:
        p_cfg = timm.models.get_pretrained_cfg(name)
        weight_tag = getattr(p_cfg, "tag", "default") or "default"
    except Exception:
        weight_tag = "default"

    best_model = md.build_model(name, pretrained=False, num_classes=9, init="scratch")
    best_ckpt = Path(cfg.out_dir) / exp_id / f"seed{cfg.seed}" / "best_model.pt"
    if best_ckpt.exists():
        best_model.load_state_dict(torch.load(best_ckpt, map_location=device, weights_only=True))

    lat_info = latency_report(best_model, batch_size=1, img_size=cfg.img_size, device=device, warmup=10, iters=50)

    row = {
        "exp_id": exp_id,
        "backbone": name,
        "family": b["family"],
        "tag": weight_tag,
        "params_m": round(res["params_m"], 2),
        "gmacs": round(res["gmacs"], 2),
        "val_macro_f1": round(res["best_val_macro_f1"], 4),
        "val_top1": round(best_val_top1, 4),
        "best_epoch": res["best_epoch"],
        "time_per_epoch_s": round(res["time_per_epoch_s"], 1),
        "latency_p50_ms": round(lat_info["p50"], 2),
        "latency_p95_ms": round(lat_info["p95"], 2),
        "throughput_img_s": round(lat_info["images_per_s"], 1),
    }
    results_backbones.append(row)
    del best_model, res
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

df_backbones = pd.DataFrame(results_backbones)
Path("runs").mkdir(exist_ok=True)
df_backbones.to_csv("runs/backbones_summary.csv", index=False)
display(df_backbones)

# Choose the next backbone using validation results only.
selected_backbone = df_backbones.sort_values("val_macro_f1", ascending=False, kind="stable").iloc[0]["backbone"]
selected_baseline_id = df_backbones.loc[df_backbones["backbone"] == selected_backbone, "exp_id"].iloc[0]
print("Selected backbone:", selected_backbone)


## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
import sys, gc, time
from pathlib import Path
import pandas as pd, torch
from train import Config, run

RECIPES = [
    {"exp_id": "T00", "axis": "Baseline", "change": "Công thức nền (Finetune, Basic Aug, CE loss)", "params": dict(init="finetune", aug="basic", loss="ce")},
    {"exp_id": "T01", "axis": "A. Khởi tạo", "change": "Đóng băng backbone (chỉ train head)", "params": dict(init="frozen", aug="basic", loss="ce")},
    {"exp_id": "T02", "axis": "B. Augmentation", "change": "Thêm ColorJitter (đổi màu)", "params": dict(init="finetune", aug="color", loss="ce")},
    {"exp_id": "T03", "axis": "C. Hàm Loss", "change": "Focal Loss (gamma=2.0)", "params": dict(init="finetune", aug="basic", loss="focal", focal_gamma=2.0)},
    {"exp_id": "T04", "axis": "Kết hợp", "change": "ColorJitter + Focal Loss", "params": dict(init="finetune", aug="color", loss="focal", focal_gamma=2.0)},
]

device = "cuda" if torch.cuda.is_available() else "cpu"
results_recipes = []
baseline_f1 = None

for r in RECIPES:
    exp_id = r["exp_id"]
    print("=" * 60)
    print(f"Chạy thí nghiệm: {exp_id} | {r['change']}")
    print("=" * 60)

    ckpt_b01 = Path("runs") / selected_baseline_id / "seed0" / "history.csv"
    if exp_id == "T00" and ckpt_b01.exists():
        df_b01 = pd.read_csv(ckpt_b01)
        best_row = df_b01.sort_values("val_macro_f1", ascending=False).iloc[0]
        val_f1 = float(best_row["val_macro_f1"])
        val_top1 = float(best_row["val_top1"])
        best_ep = int(best_row["epoch"])
        train_time = 0.0
    else:
        cfg = Config(
            exp_id=exp_id,
            backbone=selected_backbone,
            epochs=12,
            batch_size=64,   # Cấu hình chuẩn cho Kaggle
            num_workers=2,
            seed=0,
            save_test_predictions=False,
            **r["params"]
        )
        t_start = time.time()
        res = run(cfg)
        train_time = time.time() - t_start
        val_f1 = res["best_val_macro_f1"]
        best_ep = res["best_epoch"]

        hist_file = Path(cfg.out_dir) / exp_id / f"seed{cfg.seed}" / "history.csv"
        val_top1 = 0.0
        if hist_file.exists():
            df_hist = pd.read_csv(hist_file)
            row_ep = df_hist[df_hist["epoch"] == best_ep]
            if not row_ep.empty:
                val_top1 = float(row_ep["val_top1"].values[0])
        del cfg, res

    if exp_id == "T00":
        baseline_f1 = val_f1

    results_recipes.append({
        "exp_id": exp_id, "axis": r["axis"], "change": r["change"],
        "val_macro_f1": round(val_f1, 4), "val_top1": round(val_top1, 4),
        "delta_f1": round(val_f1 - baseline_f1, 4), "best_epoch": best_ep,
        "time_s": round(train_time, 1),
    })
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

df_recipes = pd.DataFrame(results_recipes)
Path("runs").mkdir(exist_ok=True)
df_recipes.to_csv("runs/training_recipe_summary.csv", index=False)
display(df_recipes)

best_recipe_id = df_recipes.sort_values("val_macro_f1", ascending=False, kind="stable").iloc[0]["exp_id"]
selected_recipe = next(r["params"].copy() for r in RECIPES if r["exp_id"] == best_recipe_id)
selected_checkpoint_id = selected_baseline_id if best_recipe_id == "T00" else best_recipe_id
print("Selected recipe:", best_recipe_id, selected_recipe)


## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
import time
from pathlib import Path
import matplotlib.pyplot as plt, numpy as np, pandas as pd, torch

from benchmark import latency_report, tta_latency
import dataset, eval as ev, inference as inf, model as md

device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt_path = Path("runs") / selected_checkpoint_id / "seed0" / "best_model.pt"

model = md.build_model(selected_backbone, pretrained=False, num_classes=9, init="scratch")
model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
model.to(device)
model.eval()

_, val_df, _ = dataset.load_split("data/labels", fold=0)
val_tf = dataset.build_transforms(train=False, img_size=224)
val_loader = dataset.make_loader(val_df, "data/images", val_tf, batch_size=64, train=False, num_workers=2)

results_inference = []

# I00: 1-view
print("[I00] Mốc 1-view...")
filenames, y_true, logits_base = inf.predict_logits(model, val_loader, device, view=inf.view_identity)
probs_base = inf.apply_temperature(logits_base, T=1.0)
m_i00 = ev.compute_metrics(y_true, probs_base.argmax(axis=1), probs_base)
lat_i00 = latency_report(model, batch_size=1, img_size=224, device=device, warmup=10, iters=50)

results_inference.append({
    "exp_id": "I00", "method": "1-view (Mốc)", "val_macro_f1": round(m_i00["macro_f1"], 4),
    "val_top1": round(m_i00["top1"], 4), "ece": round(m_i00["ece"], 4),
    "latency_p50_ms": round(lat_i00["p50"], 2), "latency_p95_ms": round(lat_i00["p95"], 2), "latency_p99_ms": round(lat_i00["p99"], 2), "relative_cost": 1.0,
})

# I01: TTA prob
print("[I01] TTA lật ngang (prob)...")
_, _, logits_flip = inf.predict_logits(model, val_loader, device, view=inf.view_hflip)
probs_tta_prob = inf.aggregate_views([logits_base, logits_flip], space="prob")
m_i01 = ev.compute_metrics(y_true, probs_tta_prob.argmax(axis=1), probs_tta_prob)
lat_i01 = tta_latency(model, k_views=2, batch_size=1, img_size=224, device=device, warmup=10, iters=50)

results_inference.append({
    "exp_id": "I01", "method": "TTA lật ngang (prob, K=2)", "val_macro_f1": round(m_i01["macro_f1"], 4),
    "val_top1": round(m_i01["top1"], 4), "ece": round(m_i01["ece"], 4),
    "latency_p50_ms": round(lat_i01["p50"], 2), "latency_p95_ms": round(lat_i01["p95"], 2), "latency_p99_ms": round(lat_i01["p99"], 2), "relative_cost": round(lat_i01["relative_cost"], 2),
})

# I03: TTA logit
print("[I03] TTA lật ngang (logit)...")
probs_tta_logit = inf.aggregate_views([logits_base, logits_flip], space="logit")
m_i03 = ev.compute_metrics(y_true, probs_tta_logit.argmax(axis=1), probs_tta_logit)
lat_i03 = tta_latency(model, k_views=2, space="logit", batch_size=1, img_size=224, device=device, warmup=10, iters=50)

results_inference.append({
    "exp_id": "I03", "method": "TTA lật ngang (logit, K=2)", "val_macro_f1": round(m_i03["macro_f1"], 4),
    "val_top1": round(m_i03["top1"], 4), "ece": round(m_i03["ece"], 4),
    "latency_p50_ms": round(lat_i03["p50"], 2), "latency_p95_ms": round(lat_i03["p95"], 2), "latency_p99_ms": round(lat_i03["p99"], 2), "relative_cost": round(lat_i03["relative_cost"], 2),
})

# I07: Temperature Scaling
print("[I07] Temperature Scaling...")
T_opt = inf.fit_temperature(logits_base, y_true)
probs_cal = inf.apply_temperature(logits_base, T_opt)
m_i07 = ev.compute_metrics(y_true, probs_cal.argmax(axis=1), probs_cal)

results_inference.append({
    "exp_id": "I07", "method": f"Temperature Scaling (T={T_opt:.2f})", "val_macro_f1": round(m_i07["macro_f1"], 4),
    "val_top1": round(m_i07["top1"], 4), "ece": round(m_i07["ece"], 4),
    "latency_p50_ms": round(lat_i00["p50"], 2), "latency_p95_ms": round(lat_i00["p95"], 2), "latency_p99_ms": round(lat_i00["p99"], 2), "relative_cost": 1.0,
})

# I08: Gộp Conv-BN
print("[I08] Gộp BatchNorm vào Conv...")
fused_model = inf.fuse_conv_bn(model)
_, fused_y, fused_logits = inf.predict_logits(fused_model, val_loader, device)
np.testing.assert_allclose(fused_logits, logits_base, rtol=1e-4, atol=1e-4)
fused_probs = inf.apply_temperature(fused_logits, 1.0)
m_fused = ev.compute_metrics(fused_y, fused_probs.argmax(axis=1), fused_probs)
lat_fused = latency_report(fused_model, batch_size=1, img_size=224, device=device, warmup=10, iters=50)

results_inference.append({
    "exp_id": "I08", "method": "Gộp Conv-BN (Fused BN)", "val_macro_f1": round(m_fused["macro_f1"], 4),
    "val_top1": round(m_fused["top1"], 4), "ece": round(m_fused["ece"], 4),
    "latency_p50_ms": round(lat_fused["p50"], 2), "latency_p95_ms": round(lat_fused["p95"], 2), "latency_p99_ms": round(lat_fused["p99"], 2),
    "relative_cost": round(lat_fused["p50"] / lat_i00["p50"], 2),
})

df_inf = pd.DataFrame(results_inference)
Path("runs").mkdir(exist_ok=True)
df_inf.to_csv("runs/inference_summary.csv", index=False)
display(df_inf)

Path("curves").mkdir(exist_ok=True)
plt.figure(figsize=(8, 5))
for r in results_inference:
    plt.scatter(r["latency_p50_ms"], r["val_macro_f1"], s=100, label=f"{r['exp_id']}: {r['method']}")
    plt.annotate(r["exp_id"], (r["latency_p50_ms"] + 0.3, r["val_macro_f1"]), fontsize=9)
plt.xlabel("Độ trễ p50 (ms, batch 1)")
plt.ylabel("Val Macro-F1")
plt.title("Đánh đổi giữa Độ chính xác và Độ trễ suy luận")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig("curves/inference_tradeoff.png", dpi=200)
plt.show()

# Select a tested deployment option on val before opening test results.
# The final runner exports the selected inference method for each seed.
selected_inference_id = df_inf.sort_values(["val_macro_f1", "ece", "latency_p95_ms"], ascending=[False, True, True], kind="stable").iloc[0]["exp_id"]
print("Validation winner:", selected_inference_id)


## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
import sys, gc, time
from pathlib import Path
import numpy as np, pandas as pd, torch

from train import Config, run
import eval as ev, inference as inf

SEEDS = [0, 1, 2]
existing_test_files = list(Path("predictions").glob("*_test.csv"))
if existing_test_files:
    raise FileExistsError("Test results already exist. Do not run the final stage again.")

# 1. Chạy Mốc T00 trên 3 seed
for seed in SEEDS:
    print(f"\n--- T00 | Seed {seed} ---")
    cfg_base = Config(
        exp_id="T00", backbone=selected_backbone, init="finetune", aug="basic", loss="ce",
        epochs=12, batch_size=64, num_workers=2, seed=seed, save_test_predictions=True,
    )
    res_base = run(cfg_base)
    del cfg_base, res_base
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

# 2. Chạy Chung kết F01 trên 3 seed
for seed in SEEDS:
    print(f"\n--- F01 | Seed {seed} ---")
    cfg_final = Config(
        exp_id="F01", backbone=selected_backbone, inference_method=selected_inference_id, **selected_recipe,
         epochs=12, batch_size=64, num_workers=2, seed=seed, save_test_predictions=True,
    )
    res_final = run(cfg_final)

    del cfg_final, res_final
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

print("\nHoàn tất tạo predictions cho eval.py!")


In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_ROOT}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_ROOT}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_ROOT}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.